# resolvers

> Container resolver registry

In [ ]:
#| default_exp resolvers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from collections import deque
from paar.resolvers import (Resolver, DictResolver, ListTupleResolver, SetResolver,
                            DequeResolver, ObjectResolver, resolve_for, register_resolver, _RESOLVERS)
class _Obj:
    def __init__(self): self.x=1; self.y='hi'

def test_dict_resolver():
    r = resolve_for({'a':1})
    assert isinstance(r, DictResolver)
    assert r.children({'a':1, 'b':2}) == [("'a'", "['a']", 1), ("'b'", "['b']", 2)]
def test_list_resolver():
    r = resolve_for([10,20])
    assert isinstance(r, ListTupleResolver)
    assert r.children([10,20]) == [('0','[0]',10), ('1','[1]',20)]
def test_tuple_uses_list_resolver():
    assert isinstance(resolve_for((1,2)), ListTupleResolver)
def test_set_resolver():
    r = resolve_for({1})
    assert isinstance(r, SetResolver)
    assert r.children({7}) == [('0','[0]',7)]
def test_deque_resolver():
    r = resolve_for(deque([10,20]))
    assert isinstance(r, DequeResolver)
    assert r.children(deque([10,20])) == [('0','[0]',10), ('1','[1]',20)]   # deque expands (was invisible before)
def test_object_resolver():
    r = resolve_for(_Obj())
    assert isinstance(r, ObjectResolver)
    assert r.children(_Obj()) == [('x','.x',1), ('y','.y','hi')]
def test_slots_object_expands():
    class S:
        __slots__=('a','b')
        def __init__(self): self.a=1; self.b=2
    r = resolve_for(S())
    assert isinstance(r, ObjectResolver)          # no __dict__, but slots make it expandable
    assert r.children(S()) == [('a','.a',1), ('b','.b',2)]
def test_property_and_class_attr_shown():
    class P:
        K = 9
        def __init__(self): self.x=1
        @property
        def dbl(self): return self.x*2
        def meth(self): return None
    kids = {n:val for n,_,val in resolve_for(P()).children(P())}
    assert kids['x']==1 and kids['dbl']==2 and kids['K']==9   # instance attr, property, class attr
    assert 'meth' not in kids                                 # methods filtered out
def test_scalars_not_containers():
    assert resolve_for(5) is None and resolve_for('hi') is None and resolve_for(None) is None
def test_primitives_not_containers():
    for x in (5, 1.5, True, 'hi', b'x', bytearray(b'x'), None, 3j):
        assert resolve_for(x) is None
def test_register_before_object_catchall():
    class _R(Resolver):
        def can(self, v): return False
    r = _R(); register_resolver(r)
    obj_i = next(i for i,x in enumerate(_RESOLVERS) if isinstance(x, ObjectResolver))
    assert _RESOLVERS.index(r) < obj_i   # inserted ahead of the catch-all
    _RESOLVERS.remove(r)
for t in [test_dict_resolver,test_list_resolver,test_tuple_uses_list_resolver,
          test_set_resolver,test_deque_resolver,test_object_resolver,test_slots_object_expands,
          test_property_and_class_attr_shown,test_scalars_not_containers,test_primitives_not_containers,
          test_register_before_object_catchall]: t()

In [ ]:
#| export
import inspect
from collections import deque

class Resolver:
    "Decides if a value is a container and yields its ordered children."
    def can(self, v): raise NotImplementedError
    def children(self, v): raise NotImplementedError   # -> list[(name, step, child)]

class DictResolver(Resolver):
    def can(self, v): return isinstance(v, dict)
    def children(self, v): return [(repr(k), f'[{k!r}]', val) for k,val in v.items()]

class ListTupleResolver(Resolver):
    def can(self, v): return isinstance(v, (list, tuple))
    def children(self, v): return [(str(i), f'[{i}]', x) for i,x in enumerate(v)]

class SetResolver(Resolver):
    "Sets: positional children in a stable order (sorted by repr); path fragment is display-only, not index-addressable."
    def can(self, v): return isinstance(v, (set, frozenset))
    def children(self, v): return [(str(i), f'[{i}]', x) for i,x in enumerate(sorted(v, key=repr))]

class DequeResolver(Resolver):
    "Deques: positional, index-addressable children (deque supports d[i])."
    def can(self, v): return isinstance(v, deque)
    def children(self, v): return [(str(i), f'[{i}]', x) for i,x in enumerate(v)]

_LEAF = (bool, int, float, complex, str, bytes, bytearray, type(None))   # never expandable

def _slot_names(v):
    "Slot names declared across v's MRO (excluding __dict__/__weakref__)."
    out = []
    for c in type(v).__mro__:
        s = getattr(c, '__slots__', ())
        if isinstance(s, str): s = (s,)
        out += [n for n in s if n not in ('__dict__', '__weakref__')]
    return out

class ObjectResolver(Resolver):
    "Catch-all like pydev's DefaultResolver: dir()-based children so __slots__ objects, properties and class attrs all expand; dunders and callables are skipped."
    def can(self, v):
        if isinstance(v, _LEAF): return False
        return bool(getattr(v, '__dict__', None)) or bool(_slot_names(v)) or hasattr(v, '__members__')
    def _keep(self, v, k):
        if k.startswith('__') and k.endswith('__'): return False
        try: a = getattr(v, k)
        except Exception: return True   # surface attribute errors as a child rather than hide the name
        return not (inspect.isroutine(a) or inspect.isbuiltin(a) or inspect.ismodule(a))
    def children(self, v):
        names = dir(v) or list(getattr(v, '__members__', ()))
        out = []
        for k in sorted(n for n in names if self._keep(v, n)):
            try: out.append((k, f'.{k}', getattr(v, k)))
            except Exception as e: out.append((k, f'.{k}', f'<error {type(e).__name__}: {e}>'))
        return out

_RESOLVERS = [DictResolver(), ListTupleResolver(), SetResolver(), DequeResolver(), ObjectResolver()]

def register_resolver(r, last=False):
    "Register a resolver; inserted before the catch-all ObjectResolver unless last=True."
    _RESOLVERS.insert(len(_RESOLVERS) if last else len(_RESOLVERS)-1, r)

def resolve_for(v):
    "First resolver that handles v, else None (None => not a container)."
    for r in _RESOLVERS:
        try:
            if r.can(v): return r
        except Exception: pass
    return None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()